# ADBahadoSingh

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.ADBahadoSingh)

class ADBahadoSingh(LinearReferenceClock):
    def postprocess(self, x):
        """Logistic transform to an Alzheimer's disease risk probability.

        The published logit constant (-0.072) is carried in the linear intercept.
        """
        return torch.sigmoid(x)



In [3]:
model = pya.models.ADBahadoSingh()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "adbahadosingh"
model.metadata["data_type"] = "DNA methylation"  # Paper: Genome-wide DNA methylation analysis
model.metadata["species"] = "Homo sapiens"  # Paper: 24 late-onset AD and 24 cognitively healthy subjects
model.metadata["year"] = 2021
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Bahado-Singh, R. O., Vishweswaraiah, S., Aydas, B., et al. (2021). Artificial intelligence and leukocyte epigenomics: Evaluation and prediction of late-onset Alzheimer's disease. PLOS ONE, 16(4), e0248375."
model.metadata["doi"] = "https://doi.org/10.1371/journal.pone.0248375"
model.metadata["notes"] = "PyAging implements the paper’s conventional four-CpG logistic-regression equation and applies a sigmoid to return LOAD case probability; it does not implement the separate high-dimensional deep-learning classifiers also evaluated in the paper."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: Genomic DNA was extracted from whole blood samples; leukocyte epigenomic biomarkers
model.metadata["predicts"] = ["late-onset Alzheimer's disease"]  # Paper: where P is Pr(y = 1|x)
model.metadata["training_target"] = ["late-onset Alzheimer's disease"]  # Paper: 24 late-onset AD and 24 cognitively healthy subjects
model.metadata["unit"] = ["probability"]  # Paper: return torch.sigmoid(x)
model.metadata["model_type"] = "logistic regression"  # Paper: The logistic regression model is represented below
model.metadata["platform"] = ["Illumina EPIC"]  # Paper: Infinium MethylationEPIC array in 24 LOAD and 24 healthy subjects
model.metadata["population"] = "older adults"  # Paper: Cases: 83.17 (7.97); Controls: 80.04 (8.42)
model.metadata["journal"] = "PLOS ONE"
model.metadata["last_author"] = "Uppala Radhakrishna"
model.metadata["n_features"] = 4
model.metadata["citations"] = 33
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o coefficients.csv https://raw.githubusercontent.com/bio-learn/biolearn/180852e2bab473303cb85da627178b1695ee9d86/biolearn/data/AD_Bahado-Singh.csv")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
mask = df['CpGmarker'].astype(str).str.lower().isin(['intercept', '(intercept)'])
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['CpGmarker'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['CoefficientTraining'].tolist()).unsqueeze(0).float()
# Published logistic-regression constant (Bahado-Singh 2021, eq. e001)
intercept = torch.tensor([-0.072]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = 'sigmoid'
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Bahado-Singh, Ray O., et al. "Artificial intelligence and '
             'leukocyte epigenomics: Evaluation and prediction of late-onset '
             'Alzheimer\'s disease." PLOS ONE 16.4 (2021): e0248375.',
 'clock_name': 'adbahadosingh',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1371/journal.pone.0248375',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2021}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: 'sigmoid'
postprocess_dependencies: None
features: ['cg02356786', 'cg04515524', 'cg00613827', 'cg07509935']
base_model_features: None

%==================================== Model Details ====================================%
Model Structure:

base_model: LinearModel(
  (linear): Linear(in_features=4, 

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
